# PRÉ-PROJETO DE PESQUISA OPERACIONAL — Python-MIP com HiGHS, CBC ou Gurobi (passos de 1 minuto)

## Otimização Operacional do Agendamento de Bombas e Distribuição de Água Tratada via Programação Linear Inteira Mista (MILP)

- Selecione o solver alterando `SOLVER` na célula de configuração.
- A primeira célula instala Python-MIP e as interfaces dos solvers.
- O horizonte é de 24 horas, discretizado em 1.440 intervalos de 1 minuto.

In [ ]:
%pip install --upgrade "mip==2.0.0" "highspy>=1.8" cbcbox gurobipy

In [ ]:
import mip
from mip.exceptions import InterfacingError
from mip import Model, xsum, BINARY, CONTINUOUS, MINIMIZE, OptimizationStatus
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Conjuntos de Índices

T = 24
horas = range(T)
MINUTOS_POR_DIA = T * 60
DURACAO_PASSO_MIN = 1
minutos = range(MINUTOS_POR_DIA)
bombas = range(1, 7)
reservatorios = range(1, 6)
setores = range(1, 13)

# 2. Topologia do Sistema (Arcos permitidos conforme o diagrama)
# Arcos (Bomba -> Reservatório)
arcos_PR = [
    (1,1), 
    (2,1), (2,2), (2,3), 
    (3,2), 
    (4,3), (4,4), 
    (5,5), 
    (6,4), (6,5)
]

# Arcos (Reservatório -> Setor de Consumo)
arcos_RD = [
    (1,1), (1,2), (1,3),
    (2,2), (2,4), (2,5),
    (3,4), (3,6), (3,7),
    (4,4), (4,8), (4,9), (4,10), (4,12),
    (5,7), (5,10), (5,11), (5,12)
]

# 3. Parâmetros Econômicos (Tarifa Horossazonal - R$/kWh)
# Tarifa fora de ponta: R$ 0.4525 | Tarifa de ponta (18h00 às 20h59 ): R$ 1.4048
C = {t: 1.4048 if 18 <= t < 21 else 0.4525 for t in horas}

# Defina False para pular o cálculo do cenário-base e seus gráficos.
GERAR_CENARIO_BASE = True

# 4. Parâmetros das bombas: (potência de referência em kW, vazão mínima e máxima em m³/h)
# Potências estimadas no ponto de vazão máxima, com 30 m de altura manométrica e 80% de eficiência global.
# O objetivo considera essa potência constante durante cada período ligado.
param_bombas = {
    1: (240, 1170, 2340),
    2: (450, 2180, 4370),
    3: (240, 1170, 2340),
    4: (290, 1400, 2810),
    5: (190,  940, 1870),
    6: (240, 1170, 2340)
}
Q_ETA_max = 9400  # m³/h; limite de vazão agregada da ETA

# 5. Parâmetros dos Reservatórios: (volume mínimo, máximo e inicial), em m³
# O volume mínimo (15%) é a reserva de segurança; o volume inicial define o estoque no início do horizonte (30%).
param_res = {
    1: (2100, 14000, 4200),
    2: (1400,  9400, 2800),
    3: (1400,  9400, 2800),
    4: (1400,  9400, 2800),
    5: ( 700,  4700, 1400)
}

# 6. Demandas-base dos setores (m³/h); total: 5195 m³/h
base_demanda = {
    1: 312,  2: 468,  3: 390,  4: 624,
    5: 390,  6: 312,  7: 546,  8: 468,
    9: 312, 10: 702, 11: 390, 12: 281
}

# Curva horária de demanda de verão; cada fator multiplica a demanda-base dos setores.
# Demanda média diária: 8312 m³/h; pico: 17663 m³/h às 18h.
curva_verao = [
    0.8, 0.5, 0.3, 0.4, 0.6, 1.0,   # 00h às 05h: menor consumo na madrugada
    1.8, 2.0, 1.9, 1.7, 1.5, 1.5,   # 06h às 11h: pico da manhã
    1.4, 1.5, 1.6, 1.7, 1.8, 2.9,   # 12h às 17h: consumo alto e crescente
    3.4, 3.0, 2.6, 2.0, 1.5, 1.0    # 18h às 23h: pico máximo às 18h
]

D = {}
for d in setores:
    for t in horas:
        D[d, t] = round(base_demanda[d] * curva_verao[t], 2)

# Vazões e demandas por intervalo de 1 minuto, em m³/min.
D_min = {(d, t): D[d, t // 60] / 60 for d in setores for t in minutos}

# Indicadores de demanda e armazenamento calculados para o horizonte diário.
demanda_horaria_total = [sum(D[d, h] for d in setores) for h in horas]
volume_demanda_diario = sum(demanda_horaria_total)
demanda_media_horaria = volume_demanda_diario / T
hora_pico = max(horas, key=lambda h: demanda_horaria_total[h])
demanda_pico_horaria = demanda_horaria_total[hora_pico]
armazenamento_util_total = sum(
    param_res[r][1] - param_res[r][0] for r in reservatorios
)

# Buffer agregado teórico com bombeamento constante igual à demanda média.
saldo_acumulado = [0]
for demanda in demanda_horaria_total:
    saldo_acumulado.append(
        saldo_acumulado[-1] + demanda_media_horaria - demanda
    )
armazenamento_util_estimado = max(saldo_acumulado) - min(saldo_acumulado)

print(f"Soma das demandas-base: {sum(base_demanda.values())} m³/h")
print(f"Demanda média horária: {demanda_media_horaria:.2f} m³/h")
print(
    f"Pico horário: {demanda_pico_horaria:.2f} m³/h às {hora_pico:02d}h"
)
print(f"Volume de demanda diário: {volume_demanda_diario:.2f} m³/dia")
print(f"Volume útil agregado dos reservatórios: {armazenamento_util_total:.0f} m³")
print(
    "Armazenamento útil estimado pelo perfil: "
    f"{armazenamento_util_estimado:.2f} m³ "
    "(estimativa agregada, com bombeamento constante na demanda média)"
)



In [ ]:
# Altere somente esta variável: "highs", "cbc" ou "gurobi".
SOLVER = "highs"
SOLVER_BACKENDS = {
    "highs": mip.HIGHS,
    "cbc": mip.CBC,
    "gurobi": mip.GRB,
}
TEMPO_LIMITE = 3600
GAP_RELATIVO = 0.003
THREADS = 0

if SOLVER not in SOLVER_BACKENDS:
    raise ValueError(
        f"SOLVER inválido: {SOLVER!r}. Escolha entre {', '.join(SOLVER_BACKENDS)}."
    )

if SOLVER == "gurobi":
    import os
    import sys
    from pathlib import Path

    gurobi_home = os.environ.get("GUROBI_HOME")
    if not gurobi_home:
        raise RuntimeError(
            "O backend Gurobi do python-mip usa a API C. Configure GUROBI_HOME "
            "para uma instalação nativa do Gurobi com licença compatível e reinicie o kernel. "
            "A licença restrita instalada pelo pacote pip gurobipy só pode ser usada pela interface gurobipy."
        )

    gurobi_home = Path(gurobi_home).expanduser()
    diretorio_bibliotecas = gurobi_home / ("bin" if os.name == "nt" else "lib")
    padrao_biblioteca = "gurobi*.dll" if os.name == "nt" else "libgurobi*"
    bibliotecas_nativas = {p.resolve() for p in diretorio_bibliotecas.glob(padrao_biblioteca)}
    if not bibliotecas_nativas:
        raise RuntimeError(
            f"Nenhuma biblioteca nativa do Gurobi ({padrao_biblioteca}) encontrada em "
            f"{diretorio_bibliotecas}. Confira o valor de GUROBI_HOME."
        )

    modulo_gurobi_mip = sys.modules.get("mip.gurobi")
    if modulo_gurobi_mip is not None:
        biblioteca_carregada = getattr(modulo_gurobi_mip, "lib_path", None)
        if biblioteca_carregada and Path(biblioteca_carregada).resolve() not in bibliotecas_nativas:
            raise RuntimeError(
                "O python-mip já carregou outra biblioteca Gurobi neste kernel. "
                "Reinicie o kernel depois de configurar GUROBI_HOME."
            )


In [ ]:
# Constrói o modelo para a otimização tarifária ou para o cenário-base.
def construir_modelo(custos, nome, vazao_ponta=None):
    try:
        m = Model(
            name=nome,
            sense=MINIMIZE,
            solver_name=SOLVER_BACKENDS[SOLVER],
        )
    except InterfacingError as exc:
        if SOLVER == "gurobi":
            raise RuntimeError(
                "O python-mip não conseguiu iniciar o Gurobi pela API C. Verifique se GUROBI_HOME "
                "aponta para a instalação nativa correta e se a licença é compatível com a API C."
            ) from exc
        raise
    m.max_seconds = TEMPO_LIMITE
    m.max_mip_gap = GAP_RELATIVO
    m.threads = THREADS
    m.verbose = 0

    x = {
        (p, t): m.add_var(var_type=BINARY, name=f"x_{p}_{t}")
        for p in bombas for t in minutos
    }
    q = {
        (p, t): m.add_var(var_type=CONTINUOUS, lb=0, name=f"q_{p}_{t}")
        for p in bombas for t in minutos
    }
    f_pr = {
        (p, r, t): m.add_var(var_type=CONTINUOUS, lb=0, name=f"f_pr_{p}_{r}_{t}")
        for p, r in arcos_PR for t in minutos
    }
    f_rd = {
        (r, d, t): m.add_var(var_type=CONTINUOUS, lb=0, name=f"f_rd_{r}_{d}_{t}")
        for r, d in arcos_RD for t in minutos
    }
    V = {
        (r, t): m.add_var(var_type=CONTINUOUS, lb=0, name=f"V_{r}_{t}")
        for r in reservatorios for t in minutos
    }

    m.objective = xsum(
        custos[t // 60] * param_bombas[p][0] * x[p, t] / 60
        for p in bombas for t in minutos
    )

    for p in bombas:
        Q_min, Q_max = param_bombas[p][1], param_bombas[p][2]
        for t in minutos:
            m += q[p, t] >= Q_min / 60 * x[p, t], f"Bomba_Min_{p}_{t}"
            m += q[p, t] <= Q_max / 60 * x[p, t], f"Bomba_Max_{p}_{t}"
            m += q[p, t] == xsum(
                f_pr[p, r, t] for pp, r in arcos_PR if pp == p
            ), f"Fluxo_Bomba_{p}_{t}"

    for t in minutos:
        m += xsum(q[p, t] for p in bombas) <= Q_ETA_max / 60, f"Capacidade_ETA_{t}"
        hora = t // 60
        if vazao_ponta is not None and hora in vazao_ponta:
            m += xsum(q[p, t] for p in bombas) == vazao_ponta[hora] / 60, f"Vazao_Ponta_{t}"

    for d in setores:
        for t in minutos:
            m += xsum(
                f_rd[r, setor, t]
                for r, setor in arcos_RD if setor == d
            ) == D_min[d, t], f"Demanda_{d}_{t}"

    for r in reservatorios:
        V_min, V_max, V_inicial = param_res[r]
        for t in minutos:
            vazao_in = xsum(
                f_pr[p, reservatorio, t]
                for p, reservatorio in arcos_PR if reservatorio == r
            )
            vazao_out = xsum(
                f_rd[reservatorio, d, t]
                for reservatorio, d in arcos_RD if reservatorio == r
            )
            volume_anterior = V_inicial if t == 0 else V[r, t - 1]
            m += (
                V[r, t] == volume_anterior
                + DURACAO_PASSO_MIN * vazao_in
                - DURACAO_PASSO_MIN * vazao_out
            ), f"Balanco_{r}_{t}"
            m += V[r, t] >= V_min, f"Vmin_{r}_{t}"
            m += V[r, t] <= V_max, f"Vmax_{r}_{t}"
        m += V[r, MINUTOS_POR_DIA - 1] >= V_inicial, f"Recuperacao_{r}"

    return m, x, q, f_pr, f_rd, V

model, x, q, f_pr, f_rd, V = construir_modelo(
    custos=C,
    nome=f"Agendamento_Bombas_{SOLVER.upper()}_MINUTOS",
)


## Cenário-base com tarifa fixa (opcional)

Defina `GERAR_CENARIO_BASE` na célula de parâmetros como `True` para executar o cenário de referência. Ele mantém as mesmas demandas, capacidades e balanços dos reservatórios, mas usa uma tarifa uniforme na função objetivo; na ponta, fixa o bombeamento total na demanda agregada horária, limitado à capacidade da ETA. O custo é então recalculado com as tarifas horárias reais. Quando desativado, a comparação e os gráficos do cenário-base são omitidos, sem afetar a otimização tarifária. Se não houver solução viável para o cenário-base, a otimização tarifária também prossegue normalmente.

In [ ]:
# O cenário-base é opcional; a otimização tarifária não depende dele.
modelo_base = None
x_base = q_base = f_pr_base = f_rd_base = V_base = None
status_base = None
consumo_base_kwh = None
custo_base_real = None
vazao_base_ponta = None
cenario_base_disponivel = False

if GERAR_CENARIO_BASE:
    horas_ponta = [h for h in horas if C[h] == max(C.values())]
    minutos_ponta = [t for t in minutos if t // 60 in horas_ponta]
    vazao_base_ponta = {
        h: min(sum(D[d, h] for d in setores), Q_ETA_max)
        for h in horas_ponta
    }
    modelo_base, x_base, q_base, f_pr_base, f_rd_base, V_base = construir_modelo(
        custos={h: 1.0 for h in horas},
        nome=f"Cenario_Base_{SOLVER.upper()}_MINUTOS",
        vazao_ponta=vazao_base_ponta,
    )
    status_base = modelo_base.optimize(max_seconds=TEMPO_LIMITE)
    cenario_base_disponivel = status_base in (
        OptimizationStatus.OPTIMAL,
        OptimizationStatus.FEASIBLE,
    )

    if not cenario_base_disponivel:
        print(
            "O cenário-base não encontrou solução viável. "
            f"Status: {status_base}. A otimização tarifária continuará normalmente."
        )
    else:
        consumo_base_kwh = sum(
            param_bombas[p][0] * x_base[p, t].x / 60
            for p in bombas for t in minutos
        )
        custo_base_real = sum(
            C[t // 60] * param_bombas[p][0] * x_base[p, t].x / 60
            for p in bombas for t in minutos
        )
        print("[CENÁRIO-BASE: TARIFA FIXA]")
        print("Status:", status_base)
        print("Vazão-alvo na ponta (m³/h):", vazao_base_ponta)
        print(f"Consumo de energia: {consumo_base_kwh:.2f} kWh")
        print(f"Custo dessa programação com as tarifas reais: R$ {custo_base_real:.2f}")
else:
    print("Cálculo do cenário-base desativado; a otimização tarifária continuará normalmente.")


In [ ]:
# Otimização com o solver selecionado na variável SOLVER
ACOMPANHAR_PROGRESSO = True
import time
model.verbose = 1
tempo_inicio = time.perf_counter()
status_terminal = model.optimize(max_seconds=TEMPO_LIMITE)
tempo_execucao = time.perf_counter() - tempo_inicio
solucao_disponivel = status_terminal in (
    OptimizationStatus.OPTIMAL,
    OptimizationStatus.FEASIBLE,
)

incumbente = model.objective_value if solucao_disponivel else None
limite_inferior = model.objective_bound if solucao_disponivel else None
gap_final = model.gap if solucao_disponivel else None
if gap_final is not None and not np.isfinite(gap_final):
    gap_final = None

print(f"\n[RESULTADOS DO {SOLVER.upper()} VIA PYTHON-MIP]")
print("Status:", status_terminal)
print(f"Custo/solução incumbente: R$ {incumbente:.2f}" if incumbente is not None else "Incumbente: não disponível")
print(f"Melhor limite inferior: R$ {limite_inferior:.2f}" if limite_inferior is not None else "Limite inferior: não disponível")
print(f"Gap relativo final: {gap_final:.2%}" if gap_final is not None else "Gap relativo final: não disponível")
print(f"Tempo: {tempo_execucao:.2f} s")

if ACOMPANHAR_PROGRESSO:
    tabela_progresso_final = pd.DataFrame([{
        "Tempo (min)": tempo_execucao / 60,
        "Melhor custo viável (R$)": incumbente,
        "Limite inferior (R$)": limite_inferior,
        "Gap (%)": 100 * gap_final if gap_final is not None else None,
    }])
    display(tabela_progresso_final.round({
        "Tempo (min)": 2,
        "Melhor custo viável (R$)": 2,
        "Limite inferior (R$)": 2,
        "Gap (%)": 3,
    }))
    print("A tabela mostra o resultado final; o backend python-mip não oferece amostragem temporal comum a todos os solvers.")

if solucao_disponivel:
    print(f"\n[SUCESSO] Solução encontrada pelo {SOLVER.upper()} via python-mip!")
else:
    print(f"\nModelo inviável ou sem solução viável encontrada pelo {SOLVER.upper()}.")


In [ ]:
# Compara a otimização com a referência, se o cenário-base foi calculado.
if solucao_disponivel:
    consumo_otimizado_kwh = sum(
        param_bombas[p][0] * x[p, t].x / 60
        for p in bombas for t in minutos
    )
    custo_otimizado_real = sum(
        C[t // 60] * param_bombas[p][0] * x[p, t].x / 60
        for p in bombas for t in minutos
    )

    linhas_comparacao = []
    if cenario_base_disponivel:
        linhas_comparacao.append({
            "Cenário": "Base (tarifa fixa)",
            "Energia (kWh)": consumo_base_kwh,
            "Custo com tarifa real (R$)": custo_base_real,
        })
    linhas_comparacao.append({
        "Cenário": "Otimização (tarifa horária)",
        "Energia (kWh)": consumo_otimizado_kwh,
        "Custo com tarifa real (R$)": custo_otimizado_real,
    })
    comparacao_custos = pd.DataFrame(linhas_comparacao)
    display(comparacao_custos)

    if cenario_base_disponivel:
        economia = custo_base_real - custo_otimizado_real
        percentual_economia = 100 * economia / custo_base_real if custo_base_real else 0
        print(f"Diferença de custo: R$ {economia:.2f} ({percentual_economia:.2f}%)")
    else:
        print("Cenário-base indisponível; a comparação de custos foi omitida.")
else:
    print("A otimização tarifária não encontrou uma solução viável para comparar.")


# GRÁFICOS E RESULTADOS

In [ ]:
# Gráficos independentes para cada cenário disponível; passo temporal de 1 minuto.
if solucao_disponivel:
    tempo_horas = [t / 60 for t in minutos]
    tempo_horas_com_fim = [t / 60 for t in range(MINUTOS_POR_DIA + 1)]
    preco_energia = [C[t // 60] for t in minutos] + [C[23]]
    cores_res = ["#4CAF50", "#8BC34A", "#CDDC39", "#FFEB3B", "#FFC107"]

    def plotar_graficos_cenario(variaveis, nome_cenario):
        vazao_total = [
            60 * sum(variaveis["q"][p, t].x for p in bombas)
            for t in minutos
        ]

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
        ax1.bar(
            tempo_horas, vazao_total, width=1 / 60, align="edge",
            color="#2196F3", alpha=0.8, edgecolor="none",
            label="Vazão bombeada (m³/h)",
        )
        ax1.set_ylabel("Vazão total (m³/h)", color="#0b5394", fontweight="bold")
        ax1.tick_params(axis="y", labelcolor="#0b5394")
        ax1.set_title(f"Perfil de bombeamento e tarifa — {nome_cenario}", fontsize=14)
        ax1_twin = ax1.twinx()
        ax1_twin.step(
            tempo_horas_com_fim, preco_energia, where="post",
            color="#f44336", linewidth=2, label="Tarifa de energia (R$/kWh)"
        )
        ax1_twin.set_ylabel("Tarifa (R$/kWh)", color="#990000", fontweight="bold")
        ax1_twin.tick_params(axis="y", labelcolor="#990000")
        ax1.axvspan(18, 21, color="red", alpha=0.1)

        for r in reservatorios:
            volume = [param_res[r][2]] + [
                variaveis["V"][r, t].x for t in minutos
            ]
            ax2.plot(
                tempo_horas_com_fim, volume, linewidth=1.5,
                color=cores_res[r - 1], label=f"Reservatório {r}"
            )
        ax2.axvspan(18, 21, color="red", alpha=0.1)
        ax2.set_xlabel("Hora do dia (0 = início, 24 = final)", fontweight="bold")
        ax2.set_ylabel("Volume armazenado (m³)", fontweight="bold")
        ax2.set_title("Dinâmica de carga e descarga dos reservatórios", fontsize=14)
        ax2.set_xticks(range(0, 25, 2))
        ax2.set_xlim(0, 24)
        ax2.legend(loc="upper left", bbox_to_anchor=(1.05, 1))
        ax1_twin.legend(loc="upper right", bbox_to_anchor=(1.25, 1.15))
        ax2.grid(True, linestyle="--", alpha=0.5)
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(12, 6))
        for r in reservatorios:
            V_max = param_res[r][1]
            volume = [param_res[r][2]] + [
                variaveis["V"][r, t].x for t in minutos
            ]
            nivel_percentual = [100 * v / V_max for v in volume]
            plt.plot(
                tempo_horas_com_fim, nivel_percentual, linewidth=1.5,
                color=cores_res[r - 1], label=f"Reservatório {r}"
            )
        plt.axvspan(
            18, 21, color="red", alpha=0.1,
            label="Horário de ponta (18h00–20h59)"
        )
        plt.axhline(100, color="black", linestyle=":", alpha=0.5)
        plt.title(f"Nível dos reservatórios (%) — {nome_cenario}", fontsize=14)
        plt.xlabel("Hora do dia (0 = início, 24 = final)", fontweight="bold")
        plt.ylabel("Volume armazenado (%)", fontweight="bold")
        plt.xticks(range(0, 25, 2))
        plt.xlim(0, 24)
        plt.ylim(0, 105)
        plt.grid(True, linestyle="--", alpha=0.7)
        plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.tight_layout()
        plt.show()

    plotar_graficos_cenario(
        {"q": q, "V": V}, "Otimização tarifária"
    )
    if cenario_base_disponivel:
        plotar_graficos_cenario(
            {"q": q_base, "V": V_base}, "Cenário-base (tarifa fixa)"
        )
else:
    print("Execute a otimização com sucesso antes de gerar os gráficos.")
